In [1]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

/home/ethanlmines/blue_dir/repos/cluster-podcast-transcription/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 2363.16it/s]


In [39]:
system_prompt = r"""
Answer with one topic and explanation per line. Separate the two with a colon. Do not add quote marks to you quotation:
topic a : transcript quote a
topic b : transcript quote b
topic c : transcript quote c
"""

orig_topics = ["politics", "business", "sports", "entertainment"]
standard_messages = [
    # {"role": "system", "content": "Only answer in comma-delimited lists with no quotations:\na,b,c."},
    {"role": "system", "content": system_prompt},
]

In [25]:
from tqdm import tqdm
import csv
import glob
import os

data_dir = "resegment_out/"

file_names = []
texts = []

# Sample one episode from each show
show_dirs = [d_path for d_path in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d_path)]
for show_dir in tqdm(show_dirs, desc="Processing shows"):
    for csv_path in glob.glob(os.path.join(show_dir, "*.csv")):
        with open(csv_path, 'r') as r:
            texts.append("".join(row['text'] for row in csv.DictReader(r)))
        file_names.append(csv_path)

Processing shows: 100%|██████████| 100/100 [00:19<00:00,  5.21it/s]


In [28]:
import re
output_pattern = re.compile('([a-z ]+) : (.+)')


In [29]:
test_string = r"""
topic a : exp a
topic b : exp b
"""
matches = output_pattern.findall(test_string)
matches

[('topic a', 'exp a'), ('topic b', 'exp b')]

In [30]:
import xgrammar as xgr
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer)

# grammar = xgr.GrammarCompiler(tokenizer_info).compile_grammar(r"""
# root ::=  () | topic ("," topic){0,2}
# topic ::= [a-z][a-z ]{0,29}
# """)

grammar = xgr.GrammarCompiler(tokenizer_info).compile_grammar(r"""
root ::=  () | entry ("\n" entry){0,2}
entry ::= [a-z][a-z ]{0,29} " : " [^\r\n]+
topic ::= [a-z][a-z ]{0,29}
""")


In [40]:

out_rows = [

]

cur_topics = sorted(orig_topics)
set_topics = set(cur_topics)
for file_path, text in zip(file_names, texts):
    user_prefix = f"""
You are tasked with performing topic modeling over podcast transcripts.
Here are the topics you've discovered so far:
{','.join(cur_topics)}

Given the following podcast transcript, generate a list of no more than 3 topics. Each topic should be on a separate line and be followed by a colon and a substring of the transcript as your justification for the topic.:
"""


    tokenized_prompt = tokenizer.apply_chat_template(standard_messages + [
        {"role": "user", "content": user_prefix + "\n" + text}
    ],
    tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    output = model.generate(**tokenized_prompt, logits_processor=[xgr.contrib.hf.LogitsProcessor(grammar)], max_new_tokens=1024)
    input_length = tokenized_prompt['input_ids'].shape[-1]
    decoded = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)

    # The quote they give must be from the text itself--avoid hallucinations
    matches = output_pattern.findall(decoded)
    print(matches)
    new_topics = [ (topic, quote) for (topic, quote) in matches if quote in text]
    set_topics = set_topics | {t for t, _ in new_topics}
    cur_topics = sorted(set_topics)

    for topic, quote in new_topics:
        out_rows.append(
            (file_path, topic, quote)
        )
    break
print(out_rows)


[('topic a', 'business'), ('topic b', 'psychedelics'), ('topic c', 'spirituality')]
[('resegment_out/6365708/6365708_00007.mp3.csv', 'topic a', 'business'), ('resegment_out/6365708/6365708_00007.mp3.csv', 'topic b', 'psychedelics'), ('resegment_out/6365708/6365708_00007.mp3.csv', 'topic c', 'spirituality')]
